In [19]:
import pandas as pd
import os
from typing import TypedDict

import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
from torch.nn.parameter import Parameter

# %matplotlib inline
import matplotlib
import numpy as np
import matplotlib.pyplot as plt

In [20]:
# Define the expected structure of your DataFrame
class MyDataSchema(TypedDict):
    Mx1: float
    Mx2: float
    Mx3: float
    My1: float
    My2: float
    # Add more columns as needed

In [21]:


# Define the path to the raw data folder
raw_data_path: str = os.path.join('data', 'raw', 'wind_speed_11_n.csv')

# Read the data
data: pd.DataFrame = pd.read_csv(raw_data_path)

In [22]:
# Load the test and train data
train_data = data.iloc[:79999]
test_data = data.iloc[80000:99999]
print(train_data.shape)

(79999, 21)


In [23]:
#Hyperparameters
num_l1 = 256
num_l2 = 256

# define network
class Net(nn.Module):

    def __init__(self, num_features, num_hidden_1, num_hidden_2, initialisation="he"):
        super(Net, self).__init__()
        # Weights
        if initialisation == "xavier":
          self.W_1 = Parameter(init.xavier_normal_(torch.Tensor(num_hidden_1, num_features)))
          self.W_2 = Parameter(init.xavier_normal_(torch.Tensor(num_hidden_2, num_hidden_1)))
          self.W_3 = Parameter(init.xavier_normal_(torch.Tensor(1, num_hidden_2)))
        else:
          self.W_1 = Parameter(init.kaiming_normal_(torch.Tensor(num_hidden_1, num_features), nonlinearity="relu"))
          self.W_2 = Parameter(init.kaiming_normal_(torch.Tensor(num_hidden_2, num_hidden_1), nonlinearity="relu"))
          self.W_3 = Parameter(init.kaiming_normal_(torch.Tensor(1, num_hidden_2), nonlinearity="relu"))


        # Biases
        self.b_1 = Parameter(init.constant_(torch.Tensor(num_hidden_1), 0))
        self.b_2 = Parameter(init.constant_(torch.Tensor(num_hidden_2), 0))
        self.b_3 = Parameter(init.constant_(torch.Tensor(1), 0))

        self.activation = torch.nn.ReLU()

    def forward(self, x):
        x = F.linear(x, self.W_1, self.b_1)
        x = self.activation(x)
        x = F.linear(x, self.W_2, self.b_2)
        x = self.activation(x)
        x = F.linear(x, self.W_3, self.b_3)
        return x



net = Net(21, num_l1, num_l2)

In [24]:
optimizer = optim.Adam(net.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [28]:
# Test the forward pass with dummy data
x = np.random.normal(0, 1, (45, 21)).astype('float32')

print(net(torch.from_numpy(x)).size())

RuntimeError: Numpy is not available